# Formulatrix Mantis

The Mantis is a liquid dispenser from Formulatrix that dispenses through exchangeable chips. PyLabRobot connects to it, identifies it, reads its firmware versions, name, motors and pressure sensors, brings the arm and Z up (a v3 against its flags and along its calibrated route to idle), moves the nozzle in the deck frame, parks it from any zone and moves the head into the waste and either wash position (v4), locks and unlocks the plate clamp (pneumatic or motor), reads which chip sits in each input slot, and shows colours on a v4's LED bar. The separate LC3 carousel is homed, rotated to an absolute angle and turned to a calibrated chip slot. Nothing in this driver switches the pump on or opens a valve.

```{device-card} formulatrix-mantis
```

```{warning}
V3 arm initialization and stop have run on one Mantis on Linux. Two consecutive setups completed with all three motors homed and no final fault bits; the operator confirmed normal arm and Z movement and the idle position. The firmware set was motion 4.20.6 and pressure 1.0.5, with boards 11 and 12 absent. The [recorded replies and command sequence](https://github.com/PyLabRobot/pylabrobot/blob/main/pylabrobot/formulatrix/mantis/driver/features/arm_tests.py) are covered by regression tests.

This driver has also run on one Mantis v4, on macOS and on Windows. The arm's bring-up, its moves, the waste, the washes and the stop that parks it have not yet run on a device through this driver; run them with the head clear and someone at the power switch. `setup()` logs a warning on a firmware set not seen on a Mantis yet. If you run this guide on a Mantis, please [report the result](../../needs-testing.md).
```

## Properties

| | |
|---|---|
| Generations | v3 and v4, told apart at setup |
| Reads | firmware versions, device name, motor limits and state, the arm's absolute encoders (v4), line pressures and the pressure board's state |
| Interface | USB, an FTDI bridge (VID 0x0403 / PID 0x6010): interface B for commands, interface A for the RFID reader |
| Sequencer | queues PPI writes and moves of the three motors, and waits until each is played; internal, for the features that act |
| Arm (v3) | homed at setup using the unit's calibration and route to idle; stopped in place at shutdown |
| Arm (v4) | brought up at setup from where its absolute encoders read; Z homed; the nozzle moved in the deck frame, across or up and down, through the unit's stage transform; parked along the unit's routes from any zone; moved into the waste and either wash position along the unit's approach paths, from wherever it stands |
| Plate clamp | v3 pneumatic lock/release with air-pump restoration; v4 motor clamp with adjustable levels |
| LC3 carousel | separate USB controller; home, rotate to an angle, select a calibrated chip slot |
| RFID | the chip in each input slot: serial, chip type, date of manufacture |
| LED bar (v4) | shows one colour, a colour per LED or an animation; lit during setup, including the arm's bring-up, and during each chip pick-up and placement; off at stop |
| Unit configuration | the clamp's lock levels, each slot's RFID antenna, the arm's calibration, its routes and the stage transform, from the unit's configuration folder or a saved file; the first two else from a recording of its generation |
| Not supported yet | switching the pump or a pressure controller, chips, dispensing |

## How it talks

The Mantis connects over USB through a two-channel FTDI bridge. Interface B carries binary frames to the device's boards: 0 for motion, 10 for pressure, 11 for the absolute position sensors (v4) and 12 for the LC3 board. The driver sends one command at a time and logs every event the device sends while connected, by what it carries. Interface A carries the RFID reader's commands.

At setup the driver reads each board's firmware version. A device whose board 11 answers is a v4; one whose board 11 does not is taken to be a v3.

## Physical setup

1. Connect the **main Mantis USB cable** to the computer and switch the Mantis on. Its USB product name is `Mantis`. The LC3 carousel has a separate connection named `LC3`; it cannot control the arm, even though it shares the same USB VID/PID. The driver filters for `Mantis`.
2. Install the FTDI dependencies: `pip install "pylabrobot[ftdi]"` (see [Installation](../../getting-started/installation.md)).
3. Close any other software connected to the Mantis. If several Mantis units are connected, pass the main controller's serial number as `device_id`.

## Unit configuration

Some of a Mantis's values are set per unit and cannot be read off the device; they sit in the unit's `.config` files, in `Data/Device/Configs`. Give setup that folder once, then save the configuration: the saved file carries the unit's values from then on, and the folder is no longer needed. So far these are the plate clamp's lock levels or, for a pneumatic clamp, its pressure-board output (`PlateClampController="PressureRegulator"`, `PlateClampAuxId="2"` in `Device.config`), each input slot's RFID antenna and, on a v4, the arm's calibration: each arm motor's offset, absolute encoder reference and encoder correction, and the routes the head takes: each zone's route to idle and the approach paths to the waste and the two wash positions; and the stage transform that places the nozzle: its transformation matrix and tilt, and the plate holder's origin.
A v3's `arm` block holds its offsets, home positions, flag searches and post-home route, from `RobotArm.config`.

Without the clamp's levels or the antennas, a recording of a unit of the same generation supplies them, with a warning: the Mantis runs, less accurately. The arm's calibration, its routes and the stage transform never come from a recording: without the calibration the arm is not brought up and setup refuses; without the routes the head is moved to idle only from near it, and the waste and the washes refuse; without the stage transform the nozzle's moves refuse. Where the folder and a saved file both hold a value, they must agree, or setup refuses. A file saved by an earlier version still loads: a value it lacks comes from the folder, else the recording, with one warning naming it.

In [ ]:
from pylabrobot.formulatrix import Mantis

mantis = Mantis(name="mantis", oem_config_directory="path/to/Data/Device/Configs")

## Setup

`setup()` opens both links, reads the firmware versions, the motor limits and the pressure sensors' calibration, takes the unit's values and sets up the RFID reader. On a v4 it then brings the arm and Z up, as in the recorded v4 startup: it energises the arm motors, declares them where their absolute encoders read, moves them to idle, homes Z, and moves to idle again. From wherever the head stands it follows its zone's routes. Without the unit's routes it does so only if the head stands near idle, in front of or behind it; from anywhere else it refuses, the arm unpowered: park the arm at idle first. `setup(skip_arm=True)` reads only and moves nothing. On a v4 the LED bar shows the setup, the arm's bring-up with it, green for a second at its end, then the idle animation.
On a v3, setup seats the arms against their stops, backs off relative to their live positions, and homes Z; repeats that sequence; homes both arm motors together, starting motor 1 before motor 0; declares the calibrated home frame and follows the unit's route to idle. Following errors during seating and homing are expected; all three motors must finish homed and fault-free. A failure or cancellation stops every motor.

`setup(skip_arm=True)` discovers the device without initializing or moving the arm. On a v3, the head pose is unavailable until initialization establishes its reference frame.

In [ ]:
await mantis.setup()
mantis.driver.configuration

### Saved configuration

Save what the device reported and the unit's values. Build the device with that file from then on, `Mantis(name="mantis", declared_configuration_json="my_mantis.json")`, with no folder: setup refuses a device that disagrees with it on the generation or the motor limits.

In [ ]:
mantis.driver.save_configuration(path="my_mantis.json")

### Simulation

`Mantis(simulation=True)` builds a Mantis with nothing behind it; nothing reaches a wire. It takes its configuration, lock levels and RFID antennas from a Mantis v4's recording (`RECORDING_MANTIS_V4`, or a file given as `declared_configuration_json`), its reads answer from it, its plate clamp's state changes at once, every declared RFID slot reads empty, and its LED bar takes every command. Its arm stands at idle, with no offsets unless the file declares the arm's calibration, and setup brings it up at once; its pump reads off, every line at 0, with the recorded sensor calibration. Its sequencer, once started, plays each queued item at once. The device name is not simulated.

For a v3 simulation, pass a saved v3 configuration containing its `arm` block. It simulates seating, homing and the declared route without opening USB; no default v3 motion calibration is supplied.

A declared pneumatic clamp simulates its valve and air-pump states. Locking starts its simulated pump if needed; release restores a pump started by the clamp.

The recording holds no zones, approach paths or stage, so the arm's moves need a unit's. Here the synthetic ones the tests declare stand in for them: with this simulated Mantis as `mantis`, the arm cells of this guide run without a device or the unit's folder.

In [ ]:
import json
import os
import tempfile

from pylabrobot.formulatrix.mantis import RECORDING_MANTIS_V4
from pylabrobot.formulatrix.mantis.driver.oem_config_tests import (
  SYNTHETIC_APPROACH_PATHS_BLOCK,
  SYNTHETIC_STAGE_BLOCK,
  SYNTHETIC_ZONES_BLOCK,
)

with open(RECORDING_MANTIS_V4, encoding="utf-8") as f:
  recording = json.load(f)
path = os.path.join(tempfile.mkdtemp(), "synthetic_mantis.json")
with open(path, "w", encoding="utf-8") as f:
  json.dump(
    {
      **recording,
      "zones": SYNTHETIC_ZONES_BLOCK,
      "approach_paths": SYNTHETIC_APPROACH_PATHS_BLOCK,
      "stage": SYNTHETIC_STAGE_BLOCK,
    },
    f,
  )
simulated = Mantis(name="simulated_mantis", simulation=True, declared_configuration_json=path)
await simulated.setup()
simulated.driver.configuration

Stop the simulated Mantis when done with it.

In [ ]:
await simulated.stop()

## Visualizer

Show the Mantis in the 3D visualizer. `start()` serves it and opens it in a browser. After each arm move the arm's links and the Z column take where the encoders read.

In [ ]:
from pylabrobot.visualizer3D import Viewer3D

viewer = Viewer3D(mantis, name="mantis_hello_world")
await viewer.start()

## Firmware versions

Each board's firmware version, by board, as setup read them; None for a board that did not answer.

In [ ]:
mantis.driver.configuration.firmware_versions

## Name

The name the motion firmware reports, `LiquidHandler` on the v4 we have run. Whether it differs between units is not known; it does not identify this Mantis.

In [ ]:
await mantis.driver.request_name()

## Motors

Motors 0 and 1 drive the arm, motor 2 drives Z. Values are in the device's units. Each motor's limits, as setup read them:

In [ ]:
mantis.arm.configuration.motor_limits

### Motor state

A motor's status word; `MOTOR_STATUS_ERROR_BITS` names the bits that report a fault:

In [ ]:
await mantis.driver.request_motor_status(motor=0)

Its demand position, where the controller drives it. It is not measured: it reads back the last position declared to the motor until the motor moves. Its servo encoder position counts only while the device is powered, from about 0 at power-on wherever the motor sits.

In [ ]:
await mantis.driver.request_motor_position(motor=0)

In [ ]:
await mantis.driver.request_encoder_position(motor=0)

Its following-error check (enabled, the largest following error), its three currents and its configuration, as the device sends them:

In [ ]:
await mantis.driver.request_following_error_configuration(motor=0)

In [ ]:
await mantis.driver.request_motor_currents(motor=0)

In [ ]:
await mantis.driver.request_motor_configuration(motor=0)

### Absolute encoders (v4)

Board 11 reads the absolute encoder of each arm motor, 0 and 1, kept across power-off. The reading is raw, not yet an angle. Each motor's first read after connecting is not valid; the driver reads again until one is.

In [ ]:
if mantis.driver.configuration.generation == "v4":
  await mantis.driver.request_absolute_encoder_position(motor=0)

## Arm

Motors 0 and 1 each turn a pair of links; the two long links meet at the head. The arm frame has its origin where motor 0 turns its links, in mm; at idle the head stands at about (15.0, 31.2). After initialization, motor positions are in degrees for the arm joints and mm down from the home flag for Z.

In [ ]:
await mantis.arm.request_joint_state()

Each arm motor's angle from its absolute encoder, corrected with the unit's calibration (**v4 only**):

In [ ]:
if mantis.driver.configuration.generation == "v4":
  await mantis.arm.request_absolute_joint_state()

In [ ]:
await mantis.arm.request_pose()

Whether the nozzle can reach a point in the deck frame, here over the plate holder's centre; nothing is clamped. A v3 initializes using the rates in its unit configuration. The v4 initialization and parking use the arm's defaults, in degrees/s and mm/s:

In [ ]:
center = mantis.plate_holder.get_location_wrt(mantis.deck, x="c", y="c")
mantis.arm.is_xy_reachable(x=center.x, y=center.y)

In [ ]:
mantis.arm.default_xy_speed, mantis.arm.default_z_speed

### Moving the arm (v4)

The nozzle moves in the deck frame, mm: `move_to_xy_position` moves it across, `move_to_z_position` up or down with the arms held. Every move across is guarded, as on the STAR: Z first rises to `minimum_traverse_height_start` (`default_minimum_traverse_height` when None, else the safe height) if the nozzle is below it. Over the plate holder the unit's stage transform places it; elsewhere the arm frame fitted to that stage. `move_to_location` also lowers Z to the location's height after. Speeds are the nozzle's, in mm/s; without one the arms move at the route profile. Every target is checked against the motor limits, and Z's speed and acceleration against the range of the recorded moves (1 to 55 mm/s, 200 to 500 mm/s²), before anything is sent. Nothing models the chip racks: move only with the path clear.

In [ ]:
pose = await mantis.arm.request_pose()
await mantis.arm.move_to_xy_position(x=pose.nozzle_location.x, y=pose.nozzle_location.y - 20.0)
await mantis.arm.park()

`park()` moves the head to idle along its zone's route, rising first if it is low; `request_is_parked()` reads whether it stands there. `recover_from_fault()` recovers after a motor error as setup does.

In [ ]:
await mantis.arm.park()
await mantis.arm.request_is_parked()

`move_to_waste()` and `move_to_wash(position=...)` follow the unit's approach paths from wherever the head stands: from one wash position to the other it stays in the wash. `park()` takes the head back out along the path it is on, then to idle.

In [ ]:
await mantis.arm.move_to_waste()
await mantis.arm.park()
await mantis.arm.move_to_wash(position=1)
await mantis.arm.park()

### Picking up a chip (v4)

Place a chip in an input slot. `pick_up_chip(slot=...)` picks it up along the unit's path into that slot, from wherever the head stands, and returns to idle; `attached_chip` names the slot the chip came from.

In [ ]:
await mantis.arm.pick_up_chip(slot=1)
mantis.arm.attached_chip

`place_chip(slot=...)` places the chip on the head back into the slot it came from, along that slot's path played backwards, and refuses any other slot; `return_chip()` does the same without naming the slot. `stop()` refuses while a chip is on the head: it stops every motor without parking, so Z is not lowered with the chip on, and stays connected with the arm brought up, so the chip can be returned. `stop(accept_chip_left_on_head=True)` stops with it there.

In [ ]:
await mantis.arm.return_chip()

## Pressure

The pump exists only when the pressure board (board 10) answered at setup. Its three pressure controllers, `"bottle"`, `"chip"` and `"vacuum"`, each regulate one line from a feedback sensor. These methods read only. The pneumatic clamp starts the air pump when needed and restores it on release; it does not change pressure-controller settings.

Each sensor's calibration, (scale, offset), as setup read it; it is never written nor saved. The offsets are re-zeroed on the device between sessions, so setup reads them every time:

In [ ]:
mantis.driver.configuration.feedback_sensor_parameters

A line's pressure, in psi; inHg on the vacuum:

In [ ]:
await mantis.pump.measure_pressure(controller="chip")

Whether the pump runs, whether a controller regulates, and its status: 0 off, 1 settled, 2 unsettled.

In [ ]:
await mantis.pump.request_pump_on()

In [ ]:
await mantis.pump.request_controller_enabled(controller="chip")

In [ ]:
await mantis.pump.request_controller_status(controller="chip")

## Plate clamp

The clamp holds the plate on the deck. Place the plate in its holder and keep the clamping area clear. `lock()` engages the clamp; `unlock()` releases it.

The v3 pneumatic clamp uses the unit's configured pressure-board output and starts the air pump when needed. It has no adjustable lock levels. The v4 motor clamp supports the unit's levels, with `lock(level=2)` selecting level 2; omitting the level uses `default_lock_level`. A failed or cancelled pneumatic lock releases the valve and restores a pump started by the clamp.

In [ ]:
await mantis.plate_clamp.lock()

Read the clamp state. On a pneumatic clamp, this is the valve's commanded state; it does not sense plate contact or confirm pneumatic pressure. On a motor clamp, it means the controller reports clamping done. The motor clamp also exposes its raw state through `request_state()`.

In [ ]:
await mantis.plate_clamp.request_locked()

### Moving over a wellplate (v4)

A well's centre in the deck frame comes from the resource model; the wellplate is taken to be clamped with A1 at the back left. Heights are those of a chip's nozzle, from the unit's calibration; where the head stands without a chip is not measured, so stay high, here 12.5 mm above the wellplate. Start from idle with the path clear.

In [ ]:
from pylabrobot.resources.corning import cor_96_wellplate_360uL_Fb

wellplate = cor_96_wellplate_360uL_Fb(name="wellplate")
mantis.plate_holder.assign_child_resource(wellplate)
top = wellplate.get_location_wrt(mantis.deck, z="t").z
await mantis.arm.park()
for name in ("A1", "H12"):
  center = wellplate.get_item(name).get_location_wrt(mantis.deck, x="c", y="c")
  await mantis.arm.move_to_xy_position(x=center.x, y=center.y)
  await mantis.arm.move_to_z_position(z=top + 12.5)
await mantis.arm.park()

Release the plate. A pneumatic clamp turns off a pump it started; a pump already running before clamping stays on. Repeating a pneumatic lock or release leaves the requested state in place.

In [ ]:
await mantis.plate_clamp.unlock()

## LC3 carousel

The LC3 chip carousel has its own USB cable and power supply. Connect it, close other software using it, and keep the Mantis head clear of its rotating area. `MantisCarousel` selects USB product `LC3` with the configured serial; it operates independently of the main Mantis. It positions the carousel but does not attach chips or coordinate the arm.

Read this unit's `Data/System/data.db`. The loader matches the serial number, saved mounting location and fitted carousel, and reads their calibration without changing the database. Supply your LC3 serial below. Missing calibration or motion settings are errors.

In [ ]:
from pylabrobot.formulatrix import CarouselConfiguration, MantisCarousel

carousel_config = CarouselConfiguration.from_oem_database(
  path="path/to/Data/System/data.db", device_id="L-000143"
)
carousel = MantisCarousel(configuration=carousel_config)

Connect and read the firmware and travel limits. Setup does not move the carousel.

In [ ]:
await carousel.setup()

Home the carousel before positioning it. `home()` does nothing when it is already homed without faults. Homing and positioning stop the motor on failure or cancellation.

In [ ]:
await carousel.home()

Select a chip slot, numbered from 1 through `carousel_config.capacity`. The angle includes the calibrated offset for slot 1. Speed is in degrees/s; omitted speed and acceleration use the unit's configured rates.

In [ ]:
await carousel.move_to_slot(1, speed=30)

Rotate to an absolute angle in degrees. Equivalent angles use the nearest permitted route, with the configured backlash compensation when approaching from a decreasing angle. Repeating the same position sends no movement command.

In [ ]:
await carousel.move_to(15, speed=30)

Read the controller's position in degrees, normalized to [0, 360). This is the commanded motor position, not an independent measurement of chip alignment.

In [ ]:
await carousel.request_position()

Return to the unit's configured idle angle.

In [ ]:
await carousel.move_to(carousel_config.idle_position, speed=30)

Close the carousel link. `stop()` stops any motion and leaves it at its current angle. The main Mantis has a separate lifecycle.

In [ ]:
await carousel.stop()

## RFID

The RFID reader reads the tag of the chip in each input slot, through that slot's antenna. A slot with no chip reads None. It exists only when the reader answered on interface A at setup; a reader that does not answer leaves setup working, without it. Each slot's antenna, as declared:

In [ ]:
mantis.rfid.slot_to_antenna

Read one slot, or every slot in turn:

In [ ]:
await mantis.rfid.request_tag(slot=1)

In [ ]:
await mantis.rfid.request_tags()

## LED bar

The v4's LED bar shows what the device does: blue while setup runs, green for a second when it is done, cyan while a chip is picked up, violet while one is placed, then the idle animation; off at stop. An action that fails leaves it red. It exists only on a v4 whose LC3 board (board 12) answered at setup; a v3 has none. Its LED count is the unit's (`NumberOfLed` in `Device.config`), as declared:

In [ ]:
mantis.led_bar.number_of_leds

Show one colour on every LED: a name (`"blue"`, `"green"`, ...), a web hex or an RGB565 value. Any animation stops first:

In [ ]:
await mantis.led_bar.set_color(color="#FF0080")

Or one colour per LED, as many as it has:

In [ ]:
await mantis.led_bar.set_color(color=["green"] * 10 + ["off"] * 10)

Each action's colour is an attribute of the bar (`setup_color`, `setup_done_color`, `error_color`, `dispense_progress_color`, `pick_up_chip_color`, `place_chip_color`), checked when it is set; each bar has its own:

In [ ]:
mantis.led_bar.setup_color = "#FF0080"

Show progress: a share of the LEDs from the left in `dispense_progress_color`, the rest off. Each frame holds the link about 62 ms, so show at most one per LED in an action:

In [ ]:
await mantis.led_bar.show_progress(done=3, total=8)

Play the idle animation, `"wave"`, repeating:

In [ ]:
await mantis.led_bar.start_animation(effect="wave")

Stop it; the bar holds what it shows:

In [ ]:
await mantis.led_bar.stop_animation()

Turn every LED off. `Mantis(show_actions=False)` shows no action's colour, only the idle animation from setup on.

In [ ]:
await mantis.led_bar.turn_off()

## Events and the sequencer

The device reports events unasked: a motor starting or ending a move, a motor error, the sequencer playing a queued item, an input changing. The driver logs each by what it carries. A motor error fails at once every wait for a motion event, with `MantisMotionError`.

The sequencer on board 0 plays queued items in order, each held for its duration in the sequencer's time steps, 1 ms on the v4. It is internal: the features that act queue through it. Here it holds the valve bank (PPI address 40) at 31, every valve closed, for about half a second: clear and start the sequencer, queue the write, then wait until the device reports it played. A write the device answers as not queued, its queue full, is sent again under a new item for at most `default_queue_full_timeout` seconds; no write is resent otherwise.

In [ ]:
await mantis.driver._unchecked_fw_clear_sequencer()
await mantis.driver._unchecked_fw_start_sequencer()
item = await mantis.driver._queue_ppi_write(duration=500, start_address=40, values=[31])
await mantis.driver._wait_for_queue_item(item=item, timeout=10)

In [ ]:
mantis.driver.default_queue_full_timeout

## Stop

Stop the visualizer and close the link. On a v4, `stop()` first aborts the sequencer and stops every motor; an arm brought up is then parked at idle and Z lowered to 20 mm, as in the recorded close; from away from idle the head first follows its zone's route, and without the unit's routes it is not parked. A v3 stays where it stopped: it is not parked, and is brought up again after reconnecting. Then the plate clamp is released.

In [ ]:
await viewer.stop()
await mantis.stop()